# Hybrid Robotic Vision: Notebook 2 — YOLO ROI Manifest Policy Sweep

This notebook is the **policy-sweep version** of Notebook 2.

Its purpose is to simulate the **lightweight onboard ROI proposal stage** while allowing you to test **multiple ROI selection policies** without changing the rest of the pipeline.

## What this notebook does

1. Load one original source video  
2. Load the corresponding encoded base-video stream  
3. Run a lightweight YOLO detector on the encoded/base video **once**  
4. Save the raw detections **once**  
5. Apply multiple ROI selector policies to the same detections  
6. For each policy:
   - filter detections
   - rank ROI candidates
   - select a limited number of ROIs per frame
   - map frame indices and coordinates back to original-video space
   - save a clean `roi_manifest.csv`
7. Save a policy comparison table

## Why this notebook exists

The detector should not have to rerun for every selector experiment.  
This notebook lets you evaluate questions like:

- how much can ROI bitrate be reduced?
- how much semantic gain is preserved?
- which selector gives the best gain per transmitted byte?

## Important design choice

This notebook **does not save crops**.

It saves only:
- raw detections
- ROI candidates for each policy
- ROI manifests for each policy
- policy-level summaries

Notebook 3 will later use one chosen policy's `roi_manifest.csv` to reconstruct crops and compress stills.

## Output layout

```text
runs/
  yolo_roi_manifest/
    <sequence_name>/
      <regime>/
        raw_detections/
          detections.csv
        policies/
          permissive/
            roi_candidates.csv
            roi_manifest.csv
            run_summary.json
          conf_size_top1/
            roi_candidates.csv
            roi_manifest.csv
            run_summary.json
          strict_small_only/
            roi_candidates.csv
            roi_manifest.csv
            run_summary.json
          balanced_top2/
            roi_candidates.csv
            roi_manifest.csv
            run_summary.json
        policy_summary.csv
```

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Define project paths and select the sequence

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/hybrid_robotic_vision")
DATA = ROOT / "data" / "uavdt"
VIDEOS = DATA / "videos"
RUNS = ROOT / "runs"

SEQUENCE_NAME = "M0101"   # <-- EDIT THIS
REGIME = "low"            # <-- choose: "low" or "moderate"

ORIGINAL_VIDEO = VIDEOS / f"{SEQUENCE_NAME}.mp4"

if REGIME == "low":
    ENCODED_VIDEO_DIR = RUNS / "uavdt_low" / "video_only" / SEQUENCE_NAME
elif REGIME == "moderate":
    ENCODED_VIDEO_DIR = RUNS / "uavdt_moderate" / "video_only" / SEQUENCE_NAME
else:
    raise ValueError("REGIME must be 'low' or 'moderate'")

ROI_RUN_DIR = RUNS / "yolo_roi_manifest" / SEQUENCE_NAME / REGIME
RAW_DET_DIR = ROI_RUN_DIR / "raw_detections"
POLICY_BASE = ROI_RUN_DIR / "policies"

for p in [RAW_DET_DIR, POLICY_BASE]:
    p.mkdir(parents=True, exist_ok=True)

print("Original video:", ORIGINAL_VIDEO)
print("Encoded video dir:", ENCODED_VIDEO_DIR)
print("Output dir:", ROI_RUN_DIR)

## 3. Find the encoded video

In [ ]:
encoded_video_candidates = [p for p in ENCODED_VIDEO_DIR.iterdir() if p.suffix.lower() in [".mp4", ".mkv", ".mov"]]
if len(encoded_video_candidates) == 0:
    raise FileNotFoundError(f"No encoded video found in {ENCODED_VIDEO_DIR}")

ENCODED_VIDEO = encoded_video_candidates[0]
print("Using encoded video:")
print(ENCODED_VIDEO)

## 4. Install packages and tools

In [ ]:
!pip install -q ultralytics opencv-python-headless pandas tqdm matplotlib
!apt-get update -qq
!apt-get install -y ffmpeg

## 5. Imports

In [ ]:
import cv2
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

## 6. Inspect video properties

In [ ]:
def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open {video_path}")
    info = {
        "path": str(video_path),
        "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "fps": float(cap.get(cv2.CAP_PROP_FPS)),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    cap.release()
    return info

orig_info = get_video_info(ORIGINAL_VIDEO)
enc_info = get_video_info(ENCODED_VIDEO)

print("Original video info:")
print(json.dumps(orig_info, indent=2))

print("\nEncoded video info:")
print(json.dumps(enc_info, indent=2))

## 7. Load YOLO detector

In [ ]:
from ultralytics import YOLO

YOLO_MODEL_NAME = "yolov8n.pt"  # change to yolov8s.pt if desired
model = YOLO(YOLO_MODEL_NAME)

print("Loaded model:", YOLO_MODEL_NAME)

## 8. Detection sweep settings

In [ ]:
START_FRAME = 0
MAX_FRAMES = 300        # set to None for all frames
FRAME_STRIDE = 5
YOLO_CONF = 0.25
YOLO_IOU = 0.45

print("Detection settings:")
print("START_FRAME =", START_FRAME)
print("MAX_FRAMES =", MAX_FRAMES)
print("FRAME_STRIDE =", FRAME_STRIDE)
print("YOLO_CONF =", YOLO_CONF)
print("YOLO_IOU =", YOLO_IOU)

## 9. Define selector policies

These policies all consume the same raw YOLO detections.

You can add more policies later without changing Notebooks 3–5.

In [ ]:
POLICIES = {
    "permissive": {
        "yolo_conf_keep": 0.20,
        "max_rois_per_frame": 3,
        "min_box_w": 0,
        "min_box_h": 0,
        "allowed_classes": None,
        "small_object_area_frac": 0.015,
        "padding_frac": 0.15,
        "priority_mode": "small_plus_lowconf",
    },

    "conf_size_top1": {
        "yolo_conf_keep": 0.40,
        "max_rois_per_frame": 1,
        "min_box_w": 12,
        "min_box_h": 12,
        "allowed_classes": ["car", "truck", "bus", "person"],
        "small_object_area_frac": 0.015,
        "padding_frac": 0.15,
        "priority_mode": "small_plus_lowconf",
    },

    "strict_small_only": {
        "yolo_conf_keep": 0.50,
        "max_rois_per_frame": 1,
        "min_box_w": 12,
        "min_box_h": 12,
        "allowed_classes": ["car", "truck", "bus", "person"],
        "small_object_area_frac": 0.020,
        "padding_frac": 0.15,
        "priority_mode": "small_only",
    },

    "balanced_top2": {
        "yolo_conf_keep": 0.40,
        "max_rois_per_frame": 2,
        "min_box_w": 12,
        "min_box_h": 12,
        "allowed_classes": ["car", "truck", "bus", "person"],
        "small_object_area_frac": 0.015,
        "padding_frac": 0.15,
        "priority_mode": "lowconf_then_small",
    },
}

print("Policies:")
for k, v in POLICIES.items():
    print("-", k, ":", v)

## 10. Run YOLO on the encoded/base video once

In [ ]:
detections = []

cap = cv2.VideoCapture(str(ENCODED_VIDEO))
if not cap.isOpened():
    raise RuntimeError(f"Could not open encoded video: {ENCODED_VIDEO}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
end_frame = total_frames if MAX_FRAMES is None else min(total_frames, START_FRAME + MAX_FRAMES)

frame_idx = 0

with tqdm(total=end_frame, desc="YOLO on encoded video") as pbar:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx >= end_frame:
            break

        if frame_idx >= START_FRAME and ((frame_idx - START_FRAME) % FRAME_STRIDE == 0):
            results = model.predict(frame, conf=YOLO_CONF, iou=YOLO_IOU, verbose=False)
            r = results[0]

            if r.boxes is not None and len(r.boxes) > 0:
                xyxy = r.boxes.xyxy.cpu().numpy()
                confs = r.boxes.conf.cpu().numpy()
                clss = r.boxes.cls.cpu().numpy().astype(int)

                for i in range(len(xyxy)):
                    x1, y1, x2, y2 = xyxy[i].tolist()
                    cls_id = int(clss[i])
                    cls_name = model.names.get(cls_id, str(cls_id))
                    detections.append({
                        "enc_frame_idx": frame_idx,
                        "enc_x1": float(x1),
                        "enc_y1": float(y1),
                        "enc_x2": float(x2),
                        "enc_y2": float(y2),
                        "conf": float(confs[i]),
                        "cls_id": cls_id,
                        "cls_name": cls_name,
                    })

        frame_idx += 1
        pbar.update(1)

cap.release()

detections_df = pd.DataFrame(detections)
detections_csv = RAW_DET_DIR / "detections.csv"
detections_df.to_csv(detections_csv, index=False)

print(f"Saved {len(detections_df)} raw detections to:")
print(detections_csv)
detections_df.head()

## 11. Quick raw-detection summary

In [ ]:
if len(detections_df) == 0:
    print("No detections found. Consider lowering YOLO_CONF or using a stronger model.")
else:
    display(detections_df["cls_name"].value_counts().to_frame("count"))

## 12. Define helper functions for policy application

In [ ]:
def map_encoded_frame_to_original(enc_frame_idx, enc_fps, orig_fps, orig_frame_count=None):
    t = enc_frame_idx / enc_fps
    orig_idx = int(round(t * orig_fps))
    if orig_frame_count is not None:
        orig_idx = max(0, min(orig_idx, orig_frame_count - 1))
    return orig_idx

def scale_box_between_frames(x1, y1, x2, y2, src_w, src_h, dst_w, dst_h):
    sx = dst_w / src_w
    sy = dst_h / src_h
    return (
        x1 * sx,
        y1 * sy,
        x2 * sx,
        y2 * sy,
    )

def apply_policy_to_detections(
    detections_df,
    policy_name,
    policy_cfg,
    enc_info,
    orig_info,
    sequence_name,
    regime,
):
    enc_w, enc_h = enc_info["width"], enc_info["height"]
    orig_w, orig_h = orig_info["width"], orig_info["height"]
    enc_fps = enc_info["fps"]
    orig_fps = orig_info["fps"]
    orig_frame_count = orig_info["frame_count"]
    enc_frame_area = enc_w * enc_h

    df = detections_df.copy()

    if len(df) == 0:
        empty_cols = [
            "policy_name", "sequence_name", "regime", "det_idx", "enc_frame_idx",
            "orig_frame_idx", "cls_id", "cls_name", "conf", "roi_reason",
            "priority_score", "area_frac", "padding_frac",
            "enc_x1", "enc_y1", "enc_x2", "enc_y2",
            "orig_x1", "orig_y1", "orig_x2", "orig_y2",
            "enc_width", "enc_height", "orig_width", "orig_height",
            "enc_fps", "orig_fps"
        ]
        return pd.DataFrame(), pd.DataFrame(columns=empty_cols)

    # Filters
    allowed = policy_cfg.get("allowed_classes")
    if allowed is not None:
        df = df[df["cls_name"].isin(allowed)].copy()

    if len(df) == 0:
        return pd.DataFrame(), pd.DataFrame()

    df["box_w"] = df["enc_x2"] - df["enc_x1"]
    df["box_h"] = df["enc_y2"] - df["enc_y1"]

    df = df[
        (df["box_w"] >= policy_cfg["min_box_w"]) &
        (df["box_h"] >= policy_cfg["min_box_h"])
    ].copy()

    df = df[df["conf"] >= policy_cfg["yolo_conf_keep"]].copy()

    if len(df) == 0:
        return pd.DataFrame(), pd.DataFrame()

    # Tags and priority
    df["area_frac"] = (df["box_w"] * df["box_h"]) / enc_frame_area
    df["is_small"] = df["area_frac"] < policy_cfg["small_object_area_frac"]

    mode = policy_cfg["priority_mode"]
    if mode == "small_plus_lowconf":
        df["priority_score"] = df["is_small"].astype(float) * 2.0 + (1.0 - df["conf"])
    elif mode == "small_only":
        df["priority_score"] = df["is_small"].astype(float) * 10.0 + 1e-6 * (1.0 - df["conf"])
    elif mode == "lowconf_then_small":
        df["priority_score"] = (1.0 - df["conf"]) * 2.0 + df["is_small"].astype(float)
    else:
        raise ValueError(f"Unknown priority mode: {mode}")

    # Per-frame selection
    selected_rows = []
    for frame_idx, group in df.groupby("enc_frame_idx"):
        selected = group.sort_values("priority_score", ascending=False).head(
            policy_cfg["max_rois_per_frame"]
        ).copy()

        if mode == "small_only":
            selected["roi_reason"] = np.where(selected["is_small"], "small_object", "fallback")
        elif mode == "lowconf_then_small":
            selected["roi_reason"] = np.where(selected["is_small"], "small_or_lowconf", "lowconf_priority")
        else:
            selected["roi_reason"] = np.where(selected["is_small"], "small_object", "priority_select")

        selected["policy_name"] = policy_name
        selected_rows.append(selected)

    if len(selected_rows) > 0:
        roi_df = pd.concat(selected_rows, ignore_index=True)
    else:
        roi_df = pd.DataFrame()

    if len(roi_df) == 0:
        return roi_df, pd.DataFrame()

    # Map to original-video space
    roi_manifest_rows = []
    for idx, row in roi_df.iterrows():
        enc_frame_idx = int(row["enc_frame_idx"])
        orig_frame_idx = map_encoded_frame_to_original(
            enc_frame_idx, enc_fps, orig_fps, orig_frame_count=orig_frame_count
        )

        ox1, oy1, ox2, oy2 = scale_box_between_frames(
            float(row["enc_x1"]), float(row["enc_y1"]),
            float(row["enc_x2"]), float(row["enc_y2"]),
            enc_w, enc_h,
            orig_w, orig_h
        )

        roi_manifest_rows.append({
            "policy_name": policy_name,
            "sequence_name": sequence_name,
            "regime": regime,
            "det_idx": int(idx),
            "enc_frame_idx": enc_frame_idx,
            "orig_frame_idx": int(orig_frame_idx),
            "cls_id": int(row["cls_id"]),
            "cls_name": row["cls_name"],
            "conf": float(row["conf"]),
            "roi_reason": row["roi_reason"],
            "priority_score": float(row["priority_score"]),
            "area_frac": float(row["area_frac"]),
            "padding_frac": float(policy_cfg["padding_frac"]),
            "enc_x1": float(row["enc_x1"]),
            "enc_y1": float(row["enc_y1"]),
            "enc_x2": float(row["enc_x2"]),
            "enc_y2": float(row["enc_y2"]),
            "orig_x1": float(ox1),
            "orig_y1": float(oy1),
            "orig_x2": float(ox2),
            "orig_y2": float(oy2),
            "enc_width": int(enc_w),
            "enc_height": int(enc_h),
            "orig_width": int(orig_w),
            "orig_height": int(orig_h),
            "enc_fps": float(enc_fps),
            "orig_fps": float(orig_fps),
        })

    roi_manifest_df = pd.DataFrame(roi_manifest_rows)
    return roi_df, roi_manifest_df

## 13. Apply all policies to the same raw detections

In [ ]:
policy_summaries = []

for policy_name, policy_cfg in POLICIES.items():
    print(f"Running policy: {policy_name}")

    roi_df, roi_manifest_df = apply_policy_to_detections(
        detections_df=detections_df,
        policy_name=policy_name,
        policy_cfg=policy_cfg,
        enc_info=enc_info,
        orig_info=orig_info,
        sequence_name=SEQUENCE_NAME,
        regime=REGIME,
    )

    policy_dir = POLICY_BASE / policy_name
    policy_dir.mkdir(parents=True, exist_ok=True)

    roi_candidates_csv = policy_dir / "roi_candidates.csv"
    roi_manifest_csv = policy_dir / "roi_manifest.csv"

    roi_df.to_csv(roi_candidates_csv, index=False)
    roi_manifest_df.to_csv(roi_manifest_csv, index=False)

    processed_frames = int(detections_df["enc_frame_idx"].nunique()) if len(detections_df) > 0 else 0
    frames_with_rois = int(roi_manifest_df["enc_frame_idx"].nunique()) if len(roi_manifest_df) > 0 else 0

    if len(roi_manifest_df) > 0 and "area_frac" in roi_manifest_df.columns:
        small_object_mask = roi_manifest_df["area_frac"] < policy_cfg["small_object_area_frac"]
        small_object_roi_ratio = float(small_object_mask.mean())
    else:
        small_object_roi_ratio = None

    summary = {
        "policy_name": policy_name,
        "num_raw_detections": int(len(detections_df)),
        "num_selected_rois": int(len(roi_manifest_df)),
        "roi_selection_ratio": float(len(roi_manifest_df) / len(detections_df)) if len(detections_df) > 0 else None,
        "processed_frames": processed_frames,
        "frames_with_rois": frames_with_rois,
        "frame_roi_coverage": float(frames_with_rois / processed_frames) if processed_frames > 0 else None,
        "small_object_roi_ratio": small_object_roi_ratio,
        "policy_cfg_json": json.dumps(policy_cfg),
        "roi_candidates_csv": str(roi_candidates_csv),
        "roi_manifest_csv": str(roi_manifest_csv),
    }

    with open(policy_dir / "run_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    policy_summaries.append(summary)

policy_summary_df = pd.DataFrame(policy_summaries)
policy_summary_df

## 14. Save the policy summary table

In [ ]:
policy_summary_csv = ROI_RUN_DIR / "policy_summary.csv"
policy_summary_df.to_csv(policy_summary_csv, index=False)

print("Saved policy summary to:")
print(policy_summary_csv)
display(policy_summary_df)

## 15. Compare policies side by side

In [ ]:
if len(policy_summary_df) == 0:
    print("No policy results available.")
else:
    comparison_cols = [
        "policy_name",
        "num_raw_detections",
        "num_selected_rois",
        "roi_selection_ratio",
        "frame_roi_coverage",
        "small_object_roi_ratio",
    ]
    display(policy_summary_df[comparison_cols].sort_values("num_selected_rois"))

## 16. Optional sanity-check visualization for one policy

This draws a few mapped boxes for the selected policy so you can verify coordinate alignment.

In [ ]:
import matplotlib.pyplot as plt

POLICY_TO_VISUALIZE = "conf_size_top1"

def read_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None
    return frame

def draw_box_rgb(frame_bgr, box, color=(255, 0, 0), label=None):
    img = cv2.cvtColor(frame_bgr.copy(), cv2.COLOR_BGR2RGB)
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    if label:
        cv2.putText(img, label, (x1, max(y1 - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

vis_manifest_csv = POLICY_BASE / POLICY_TO_VISUALIZE / "roi_manifest.csv"
if not vis_manifest_csv.exists():
    print("No ROI manifest found for visualization.")
else:
    vis_df = pd.read_csv(vis_manifest_csv)

    if len(vis_df) == 0:
        print("Selected policy has no ROI rows.")
    else:
        n_show = min(4, len(vis_df))
        sample_rows = vis_df.head(n_show)

        plt.figure(figsize=(14, 4 * n_show))
        for i, (_, row) in enumerate(sample_rows.iterrows(), 1):
            enc_frame = read_frame(ENCODED_VIDEO, int(row["enc_frame_idx"]))
            orig_frame = read_frame(ORIGINAL_VIDEO, int(row["orig_frame_idx"]))

            enc_box = (row["enc_x1"], row["enc_y1"], row["enc_x2"], row["enc_y2"])
            orig_box = (row["orig_x1"], row["orig_y1"], row["orig_x2"], row["orig_y2"])

            enc_vis = draw_box_rgb(enc_frame, enc_box, label=f'{row["cls_name"]}:{row["conf"]:.2f}')
            orig_vis = draw_box_rgb(orig_frame, orig_box, label=f'{row["cls_name"]}:{row["conf"]:.2f}')

            plt.subplot(n_show, 2, 2 * i - 1)
            plt.imshow(enc_vis)
            plt.title(f'{POLICY_TO_VISUALIZE} | Encoded frame {int(row["enc_frame_idx"])}')
            plt.axis("off")

            plt.subplot(n_show, 2, 2 * i)
            plt.imshow(orig_vis)
            plt.title(f'{POLICY_TO_VISUALIZE} | Original frame {int(row["orig_frame_idx"])}')
            plt.axis("off")

        plt.tight_layout()
        plt.show()

## 17. Save a global run summary

This JSON summarizes the detector run and the policy sweep.

In [ ]:
global_summary = {
    "sequence_name": SEQUENCE_NAME,
    "regime": REGIME,
    "original_video": str(ORIGINAL_VIDEO),
    "encoded_video": str(ENCODED_VIDEO),
    "yolo_model": YOLO_MODEL_NAME,
    "start_frame": START_FRAME,
    "max_frames": MAX_FRAMES,
    "frame_stride": FRAME_STRIDE,
    "yolo_conf": YOLO_CONF,
    "yolo_iou": YOLO_IOU,
    "num_raw_detections": int(len(detections_df)),
    "detections_csv": str(detections_csv),
    "policy_summary_csv": str(policy_summary_csv),
    "policy_names": list(POLICIES.keys()),
}

global_summary_path = ROI_RUN_DIR / "run_summary.json"
with open(global_summary_path, "w") as f:
    json.dump(global_summary, f, indent=2)

print("Saved global summary to:")
print(global_summary_path)
print(json.dumps(global_summary, indent=2))

## 18. What this notebook completed

At this point you now have:
- one shared raw detection file
- one policy sweep over multiple ROI selectors
- one `roi_manifest.csv` per policy
- a policy summary table you can use to choose which policy to feed into Notebook 3

## How to use this with Notebook 3

For Notebook 3, choose a policy and point the ROI-manifest path to:

```text
runs/yolo_roi_manifest/<sequence>/<regime>/policies/<policy_name>/roi_manifest.csv
```

Examples:
- `permissive`
- `conf_size_top1`
- `strict_small_only`
- `balanced_top2`

## Recommended next step

Run Notebook 3–5 for at least two policies:
- `permissive`
- `conf_size_top1`

Then compare:
- ROI bitrate
- mean confidence gain
- confidence gain per KB